# Kairos: actual14B adapter resource pilot

Select an available free T4 GPU. This notebook downloads the pinned9.93GB NF4 checkpoint and updates its adapter weights. It uses only synthetic source-edit fixtures to measure memory, gradients and interruption/resume. It does not claim a trained repair agent or improved repair scores.

Zero API spend. No Drive mount, credentials, private repository or real investigation records are uploaded. Temporary cloud scratch is bounded to20GiB; keep4GiB free. The Mac stays out of large-model training.7B is never substituted.


## Install in an isolated environment

Preserve the runtime Torch/CUDA stack and pin the training packages. A dependency conflict stops before model download. Installation logs remain available.


In [ ]:
import os,sys,json,subprocess,venv,shutil,platform,importlib.metadata
from pathlib import Path
ROOT=Path('/content/kairos14-resource014')
if platform.system()!='Linux':raise RuntimeError('Use an isolated Colab/Kaggle Linux GPU runtime. Do not run on the Mac.')
if ROOT.exists():raise RuntimeError('Existing pilot preserved. Do not overwrite or automatically rerun it.')
if shutil.disk_usage('/content').free<24*1024**3:raise RuntimeError('Pilot requires20GiB temporary capacity plus4GiB reserve')
ROOT.mkdir(mode=0o700);(ROOT/'tmp').mkdir()
(ROOT/'owner.json').write_text(json.dumps({'purpose':'actual14B synthetic resource pilot','temporary_scratch_ceiling_bytes':20*1024**3,'minimum_free_bytes':4*1024**3,'spend':0,'private_data_uploaded':False}))
venv.EnvBuilder(system_site_packages=True,with_pip=True).create(ROOT/'venv')
PYTHON=ROOT/'venv/bin/python'
constraints=[]
for package in ['torch','torchvision','triton']:
 try:constraints.append(package+'=='+importlib.metadata.version(package))
 except importlib.metadata.PackageNotFoundError:pass
constraints.extend(['transformers==4.57.6','peft==0.21.2','accelerate==1.15.0','bitsandbytes==0.50.2','huggingface-hub==0.36.2','datasets==4.3.0','trl==0.24.0'])
(ROOT/'constraints.txt').write_text('\n'.join(constraints)+'\n')
env={**os.environ,'TMPDIR':str(ROOT/'tmp'),'PIP_CONFIG_FILE':'/dev/null','PIP_DISABLE_PIP_VERSION_CHECK':'1','HF_HOME':str(ROOT/'hf-cache'),'XDG_CACHE_HOME':str(ROOT/'cache')}
with (ROOT/'installation.log').open('x') as log:
 result=subprocess.run([str(PYTHON),'-m','pip','install','--no-cache-dir','--index-url','https://pypi.org/simple','-c',str(ROOT/'constraints.txt'),'unsloth==2026.10.6','unsloth_zoo==2026.10.6','transformers==4.57.6','peft==0.21.2','accelerate==1.15.0','bitsandbytes==0.50.2','huggingface-hub==0.36.2','datasets==4.3.0','trl==0.24.0'],stdout=log,stderr=subprocess.STDOUT,env=env,timeout=1200)
if result.returncode:raise RuntimeError('Pinned installation failed. Preserve installation.log; no model training started.')
packages=subprocess.check_output([str(PYTHON),'-m','pip','list','--format=json'],env=env,text=True)
(ROOT/'runtime-packages.json').write_text(packages)
print('Installation finished. No weights or training have started yet.')


## Run actual14B QLoRA updates

Rank8 on q_proj/v_proj, alpha16, dropout0 for the optimized resource profile, FP16,2048 padded sequence, microbatch1 and accumulation8.40 total optimizer updates across full/interrupted/resumed conditions. Maximum run90minutes. Loss applies only to assistant target tokens. Outputs are captured in training.log; no public model endpoint is started.


In [ ]:
import hashlib,os,subprocess
SCRIPT='"""Actual14B QLoRA resource pilot on synthetic fixtures; no repair-score claim."""\nimport os,json,hashlib,gc,math,shutil,time\nfrom pathlib import Path\nROOT=Path(os.environ.get(\'KAIROS_PILOT_ROOT\',\'/content/kairos14-resource014\'));OUT=ROOT/\'results\';MODEL=ROOT/\'model\'\nREPO=\'unsloth/Qwen2.5-Coder-14B-Instruct-bnb-4bit\';REV=\'ecfc63c083900ce4f19daad695e711f38ab98fcf\'\nWEIGHTS={\'model-00001-of-00002.safetensors\':(\'8fbefb1b257e9443b3b817b655449f1c2884afefcad1a5a6c33f9097169e383e\',4998641583),\'model-00002-of-00002.safetensors\':(\'d5c3d547462b12bac817a5af131be0862220b67ec7a7114b07fdf810df6ac285\',4933695760)}\nos.environ.update(HF_HOME=str(ROOT/\'hf-cache\'),HF_HUB_DISABLE_IMPLICIT_TOKEN=\'1\',HF_HUB_DISABLE_XET=\'1\',TORCH_HOME=str(ROOT/\'torch-cache\'),XDG_CACHE_HOME=str(ROOT/\'cache\'),CUDA_CACHE_PATH=str(ROOT/\'cuda-cache\'),WANDB_DISABLED=\'true\',CUBLAS_WORKSPACE_CONFIG=\':4096:8\',TOKENIZERS_PARALLELISM=\'false\',UNSLOTH_DISABLE_STATISTICS=\'1\')\ndef sha(path):\n h=hashlib.sha256()\n with path.open(\'rb\') as f:\n  for chunk in iter(lambda:f.read(1024*1024),b\'\'):h.update(chunk)\n return h.hexdigest()\ndef guard():\n size=sum(p.stat().st_size for p in ROOT.rglob(\'*\') if p.is_file() and not p.is_symlink())\n if size>20*1024**3 or shutil.disk_usage(ROOT).free<4*1024**3:raise RuntimeError(\'20GiB pilot scratch or4GiB reserve exceeded\')\n return size\ndef write(name,data):\n p=OUT/name\n with p.open(\'x\') as f:json.dump(data,f,indent=2);f.write(\'\\n\');f.flush();os.fsync(f.fileno())\ndef tree_cpu(value):\n if torch.is_tensor(value):return value.detach().cpu().clone()\n if isinstance(value,dict):return {k:tree_cpu(v) for k,v in value.items()}\n if isinstance(value,list):return [tree_cpu(v) for v in value]\n if isinstance(value,tuple):return tuple(tree_cpu(v) for v in value)\n return value\ndef equal(a,b):\n if torch.is_tensor(a):return torch.is_tensor(b) and torch.equal(a,b)\n if type(a)!=type(b):return False\n if isinstance(a,dict):return a.keys()==b.keys() and all(equal(a[k],b[k]) for k in a)\n if isinstance(a,(list,tuple)):return len(a)==len(b) and all(equal(x,y) for x,y in zip(a,b))\n return a==b\ndef main():\n global torch\n import platform\n if platform.system()!=\'Linux\':raise RuntimeError(\'Use an isolated Linux GPU runtime; never train this model on the Mac\')\n ROOT.mkdir(exist_ok=True);OUT.mkdir(exist_ok=True)\n with (ROOT/\'pilot.started\').open(\'x\') as f:f.write(\'No automatic rerun after an interrupted dispatch. Preserve results/checkpoints.\\n\')\n try:\n  guard()\n  # Import Unsloth before Transformers so its documented memory patches apply.\n  from unsloth import FastLanguageModel\n  import torch,importlib.metadata\n  from huggingface_hub import snapshot_download\n  from transformers import Trainer,TrainerCallback,TrainingArguments,DataCollatorForSeq2Seq,set_seed\n  from peft import get_peft_model_state_dict,set_peft_model_state_dict\n  if not torch.cuda.is_available():raise RuntimeError(\'CUDA GPU required; no CPU or smaller-model fallback\')\n  runtime={n:importlib.metadata.version(n) for n in [\'unsloth\',\'unsloth_zoo\',\'torch\',\'transformers\',\'peft\',\'accelerate\',\'bitsandbytes\']}\n  if torch.cuda.get_device_capability(0)<(7,5):raise RuntimeError(\'This reviewed pilot requires CUDA compute capability7.5 or newer\')\n  write(\'intent.json\',{\'version\':\'gpu14-resource-intent014-v1\',\'model\':REPO,\'revision\':REV,\'runtime\':runtime,\'device\':torch.cuda.get_device_name(0),\'gpu_total_bytes\':torch.cuda.get_device_properties(0).total_memory,\'synthetic_fixture_only\':True,\'repair_training_corpus_admitted\':False,\'planned_optimizer_updates\':40,\'conditions\':\'Full20, interrupted10, resumed10 additional updates\',\'context_tokens\':2048,\'microbatch\':1,\'gradient_accumulation\':8,\'adapter_rank\':8,\'adapter_dropout\':0.0,\'adapter_targets\':[\'q_proj\',\'v_proj\'],\'learning_rate\':0.0001,\'quantization\':\'NF4 prequantized derivative\',\'checkpointing\':\'Unsloth memory-efficient checkpointing\',\'spend\':\'No API, subscription purchase or paid endpoint\',\'script_sha256\':sha(Path(__file__))})\n  print(\'Downloading pinned14B NF4 weights,9,932,337,343bytes; no credentials required.\',flush=True)\n  snapshot_download(REPO,revision=REV,local_dir=str(MODEL),allow_patterns=[\'*.json\',\'*.safetensors\',\'*.model\',\'*.txt\',\'*.jinja\'],token=False,max_workers=1)\n  for name,(expected,size) in WEIGHTS.items():\n   p=MODEL/name\n   if p.stat().st_size!=size or sha(p)!=expected:raise RuntimeError(\'Pinned model weight bytes differ\')\n  guard();write(\'weight-receipt.json\',{\'model\':REPO,\'revision\':REV,\'weights\':{n:{\'sha256\':h,\'bytes\':s} for n,(h,s) in WEIGHTS.items()},\'files\':{p.relative_to(MODEL).as_posix():{\'sha256\':sha(p),\'bytes\':p.stat().st_size} for p in MODEL.rglob(\'*\') if p.is_file() and \'.cache\' not in p.parts}})\n  os.environ[\'HF_HUB_OFFLINE\']=\'1\';set_seed(42);torch.use_deterministic_algorithms(True)\n  model,tokenizer=FastLanguageModel.from_pretrained(model_name=str(MODEL),max_seq_length=2048,dtype=torch.float16,load_in_4bit=True,local_files_only=True,trust_remote_code=False)\n  model=FastLanguageModel.get_peft_model(model,r=8,target_modules=[\'q_proj\',\'v_proj\'],lora_alpha=16,lora_dropout=0.0,bias=\'none\',use_gradient_checkpointing=\'unsloth\',random_state=42,use_rslora=False,loftq_config=None)\n  if any(\'lora_\' not in n for n,p in model.named_parameters() if p.requires_grad):raise RuntimeError(\'Non-adapter parameter is trainable\')\n  if tokenizer.pad_token_id is None:tokenizer.pad_token=tokenizer.eos_token\n  tokenizer.padding_side=\'right\';initial=tree_cpu(get_peft_model_state_dict(model));data=[]\n  for i in range(160):\n   value=i%20+1;original=f\'def value():\\n    return {value-1}\\n\';handle=\'S\'+hashlib.sha256(original.encode()).hexdigest()[:12]\n   prompt=json.dumps({\'issue\':f\'value() must return {value}.\',\'original_source\':[{\'source_id\':handle,\'path\':\'package/value.py\',\'start_line\':1,\'end_line\':2,\'numbered_original_source\':\'1: def value():\\n2:     return \'+str(value-1)+\'\\n\'}]},sort_keys=True)\n   target=json.dumps({\'action\':\'propose_patch\',\'edits\':[{\'source_id\':handle,\'start_line\':2,\'end_line\':2,\'new\':f\'    return {value}\\n\'}]},sort_keys=True)\n   messages=[{\'role\':\'system\',\'content\':\'Return one source-handle JSON edit action.\'},{\'role\':\'user\',\'content\':prompt}]\n   prefix=tokenizer.apply_chat_template(messages,tokenize=True,add_generation_prompt=True)\n   full=tokenizer.apply_chat_template(messages+[{\'role\':\'assistant\',\'content\':target}],tokenize=True,add_generation_prompt=False)\n   if full[:len(prefix)]!=prefix or len(full)>2048:raise RuntimeError(\'Exact target mask or sequence budget failed\')\n   data.append({\'input_ids\':full,\'attention_mask\':[1]*len(full),\'labels\':[-100]*len(prefix)+full[len(prefix):]})\n  collator=DataCollatorForSeq2Seq(tokenizer,model=None,padding=\'max_length\',max_length=2048,label_pad_token_id=-100)\n  class Capture(TrainerCallback):\n   def __init__(self,name,stop):self.name=name;self.stop=stop;self.logs=[]\n   def on_log(self,args,state,control,logs=None,**kwargs):\n    if logs:\n     item={\'step\':state.global_step,**{k:float(logs[k]) for k in [\'loss\',\'grad_norm\'] if k in logs}}\n     if any(not math.isfinite(v) for k,v in item.items()):raise RuntimeError(\'Nonfinite loss/gradient\')\n     self.logs.append(item)\n     with (OUT/(self.name+\'-metrics.jsonl\')).open(\'a\') as f:f.write(json.dumps(item)+\'\\n\');f.flush();os.fsync(f.fileno())\n     print(self.name,item,flush=True)\n   def on_step_end(self,args,state,control,**kwargs):\n    if self.stop and state.global_step>=self.stop:control.should_save=True;control.should_training_stop=True\n    return control\n   def on_save(self,args,state,control,**kwargs):\n    folder=Path(args.output_dir)/(\'checkpoint-\'+str(state.global_step))\n    manifest={p.relative_to(folder).as_posix():{\'sha256\':sha(p),\'bytes\':p.stat().st_size} for p in folder.rglob(\'*\') if p.is_file()}\n    (folder/\'checkpoint-files.json\').write_text(json.dumps(manifest,indent=2));guard()\n  def training(name,stop=None,resume=None):\n   set_seed(42);set_peft_model_state_dict(model,initial);FastLanguageModel.for_training(model)\n   callback=Capture(name,stop);folder=OUT/name\n   args=TrainingArguments(output_dir=str(folder),max_steps=20,per_device_train_batch_size=1,gradient_accumulation_steps=8,learning_rate=0.0001,fp16=True,bf16=False,optim=\'adamw_torch\',logging_steps=1,logging_nan_inf_filter=False,save_strategy=\'no\',report_to=\'none\',push_to_hub=False,seed=42,data_seed=42,dataloader_num_workers=0,dataloader_pin_memory=False,remove_unused_columns=False,disable_tqdm=True)\n   if resume:\n    captured=json.loads((resume/\'checkpoint-files.json\').read_text())\n    for path,item in captured.items():\n     if \'..\' in Path(path).parts or Path(path).is_absolute():raise RuntimeError(\'Unsafe checkpoint path\')\n     if sha(resume/path)!=item[\'sha256\']:raise RuntimeError(\'Resume checkpoint changed\')\n   torch.cuda.reset_peak_memory_stats();started=time.monotonic()\n   trainer=Trainer(model=model,args=args,train_dataset=data,data_collator=collator,callbacks=[callback]);trainer.train(resume_from_checkpoint=str(resume) if resume else None);torch.cuda.synchronize()\n   result={\'adapter\':tree_cpu(get_peft_model_state_dict(model)),\'optimizer\':tree_cpu(trainer.optimizer.state_dict()),\'scheduler\':tree_cpu(trainer.lr_scheduler.state_dict()),\'cpu_rng\':torch.get_rng_state().clone(),\'cuda_rng\':torch.cuda.get_rng_state(0).clone(),\'steps\':trainer.state.global_step,\'seconds\':time.monotonic()-started,\'peak_reserved_bytes\':torch.cuda.max_memory_reserved(),\'logs\':callback.logs}\n   target=folder/\'final-adapter\';model.save_pretrained(str(target),safe_serialization=True)\n   for config in target.glob(\'adapter_config.json\'):\n    d=json.loads(config.read_text());d[\'base_model_name_or_path\']=REPO;d[\'revision\']=REV;config.write_text(json.dumps(d,indent=2)+\'\\n\')\n   write(name+\'-receipt.json\',{k:v for k,v in result.items() if k not in [\'adapter\',\'optimizer\',\'scheduler\',\'cpu_rng\',\'cuda_rng\',\'logs\']})\n   del trainer;gc.collect();torch.cuda.empty_cache();guard();return result\n  print(\'Starting actual14B adapter updates on explicitly synthetic resource fixtures.\',flush=True)\n  full=training(\'full\');partial=training(\'interrupted\',stop=10);del partial;gc.collect();torch.cuda.empty_cache()\n  resumed=training(\'resumed\',resume=OUT/\'interrupted/checkpoint-10\')\n  checks={k:equal(full[k],resumed[k]) for k in [\'adapter\',\'optimizer\',\'scheduler\',\'cpu_rng\',\'cuda_rng\']}\n  peak=max(full[\'peak_reserved_bytes\'],resumed[\'peak_reserved_bytes\']);total=torch.cuda.get_device_properties(0).total_memory\n  checks.update(adapter_updated=not equal(initial,full[\'adapter\']),twenty_steps_completed=full[\'steps\']==resumed[\'steps\']==20,nonzero_gradient=any(x.get(\'grad_norm\',0)>0 for x in full[\'logs\']),vram_headroom=1-peak/total>=0.15)\n  write(\'receipt.json\',{\'version\':\'gpu14-resource-result014-v1\',\'model\':REPO,\'revision\':REV,\'runtime\':runtime,\'checks\':checks,\'passed\':all(checks.values()),\'peak_reserved_bytes\':peak,\'gpu_total_bytes\':total,\'headroom_fraction\':1-peak/total,\'trainable_parameters\':sum(p.numel() for p in model.parameters() if p.requires_grad),\'synthetic_fixture_only\':True,\'repair_quality_verified\':False,\'full_repair_training_complete\':False,\'scope\':\'Actual pretrained14B adapter updates and resume on synthetic source-edit fixtures. No independent repair-effectiveness or qualified-corpus training claim.\'})\n  print(\'GPU14_RESOURCE_RESULT\',json.dumps(json.loads((OUT/\'receipt.json\').read_text())),flush=True)\n except BaseException as error:\n  if not (OUT/\'failure.json\').exists():write(\'failure.json\',{\'version\':\'gpu14-resource-failure014-v1\',\'failure_type\':type(error).__name__,\'message\':str(error)[:1000],\'repair_quality_verified\':False,\'full_repair_training_complete\':False})\n  raise\n\nif __name__==\'__main__\':main()\n'
script_path=ROOT/'gpu14_pilot.py'
script_path.write_text(SCRIPT)
assert hashlib.sha256(script_path.read_bytes()).hexdigest()=='d131fdeff3a5e5d15fe05735f7645b9af8f208cfdc654bd09c9d9e423a5ad664'
env={**os.environ,'KAIROS_PILOT_ROOT':str(ROOT),'TMPDIR':str(ROOT/'tmp'),'HF_HOME':str(ROOT/'hf-cache'),'XDG_CACHE_HOME':str(ROOT/'cache')}
print('Starting14B. Full20, interrupted10 and resumed10 additional optimizer updates. Synthetic fixtures only.',flush=True)
with (ROOT/'training.log').open('x') as log:
 result=subprocess.run([str(PYTHON),'-u',str(script_path)],stdout=log,stderr=subprocess.STDOUT,env=env,timeout=5400)
print('Exit status:',result.returncode)
print((ROOT/'training.log').read_text()[-6000:])
if result.returncode:raise RuntimeError('14B pilot failed; preserve artifacts. No smaller model was substituted.')


## Export evidence, including failures

Run even if the pilot failed. Export only adapter/checkpoint/log artifacts, never base-model weights or account state. Verify the download before disconnecting/deleting this owned runtime. Then move this notebook to the recoverable bin, sign out and remove the remembered account from the chooser. Provider-retained logs are outside this cleanup.


In [ ]:
import zipfile,hashlib,json
from google.colab import files
archive=ROOT/'kairos14-resource014.zip'
selected=[]
if (ROOT/'results').exists():selected.extend(p for p in (ROOT/'results').rglob('*') if p.is_file())
selected.extend(p for p in [ROOT/'owner.json',ROOT/'constraints.txt',ROOT/'installation.log',ROOT/'runtime-packages.json',ROOT/'gpu14_pilot.py',ROOT/'training.log'] if p.is_file())
if not selected:raise RuntimeError('No captured artifacts to export')
if any(p.is_symlink() or not p.resolve().is_relative_to(ROOT.resolve()) for p in selected):raise RuntimeError('Artifact path escapes the owned pilot')
raw_bytes=sum(p.stat().st_size for p in selected)
if raw_bytes>192*1024**2:raise RuntimeError('Artifact export exceeds192MiB bound; preserve runtime for inspected packaging')
manifest={'version':'gpu14-resource-export014-v1','synthetic_fixture_only':True,'full_repair_training_complete':False,'files':{p.relative_to(ROOT).as_posix():{'sha256':hashlib.file_digest(p.open('rb'),'sha256').hexdigest(),'bytes':p.stat().st_size} for p in selected}}
with zipfile.ZipFile(archive,'x',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
 for p in selected:z.write(p,p.relative_to(ROOT).as_posix())
 z.writestr('manifest.json',json.dumps(manifest,indent=2))
print({'artifact_bytes':archive.stat().st_size,'sha256':hashlib.file_digest(archive.open('rb'),'sha256').hexdigest(),'contains_base_weights':False})
files.download(str(archive))
